Cell 1: Universal Set-Up

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import sklearn
import os
import yfinance as yf
import pandas_datareader.data as web
import time
import requests
import plotly.express as px  # Added for interactive hover
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from google.colab import drive
from datetime import datetime
from scipy.stats import norm
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# 1. Mount Drive
drive.mount('/content/drive')

# 2. Setup Standard Directory
base_dir = '/content/drive/MyDrive/MSFin_Python'
sub_folders = ['data', 'notebooks', 'output']

for folder in sub_folders:
    os.makedirs(os.path.join(base_dir, folder), exist_ok=True)

print(f"Environment Ready: Drive mounted and folders verified at {base_dir}")

Cell 2: File Configuration

In [ ]:
# --- CSV FILE NAME FOR THIS ANALYSIS ---
SOURCE_FILENAME = 'L12_Employment_Data_Finance.csv'

# --- AUTOMATED PATH BUILDING ---
data_dir = os.path.join(base_dir, 'data')
input_file = os.path.join(data_dir, SOURCE_FILENAME)

# Load and initial cleaning (Remove 2020 as requested)
df = pd.read_csv(input_file)
df = df[df['year'] != 2020].copy()

print(f"Analysis Ready: {len(df)} records loaded (Excluding 2020)")

Cell 3: Demonstrating DataFrame Elements

In [ ]:
# 1. Boolean Masking: Find High-Wage Finance Quarters
high_wage_fin = df[(df['industry'] == 'Finance') & (df['median_wage'] > 100000)]

# 2. Groupby: Average sample size by State
state_reliability = df.groupby('state')['sample_size'].mean()

# 3. Calculated Columns: Real Wage Premium ($)
# First, we need to pivot to align Finance and Other side-by-side
pivot_df = df.pivot_table(index=['year', 'state'], columns='industry', values='median_wage').reset_index()
pivot_df['Wage_Premium'] = pivot_df['Finance'] - pivot_df['Other']

# 4. Merge: Bringing Total Employment back into our pivot
employment_totals = df.groupby(['year', 'state'])['total_est_empl'].sum().reset_index()
final_demo_df = pd.merge(pivot_df, employment_totals, on=['year', 'state'])

print("DataFrame manipulation demonstration complete.")
display(final_demo_df.head().style.format({
    'Finance': '${:,.0f}',
    'Other': '${:,.0f}',
    'Wage_Premium': '${:,.0f}',
    'total_est_empl': '{:,.0f}' # Added format for total_est_empl
}))

Cell 4: Weighted Median and Comparison Tool

In [ ]:
from ipywidgets import interact, widgets

def get_weighted_median(data):
    # Simplified weighted median: Sum of (Median * Weight) / Total Weight
    return (data['median_wage'] * data['total_est_empl']).sum() / data['total_est_empl'].sum()

def compare_states(state1, state2):
    comparison_states = [state1, state2]
    sub = df[df['state'].isin(comparison_states)].copy()

    # Calculate weighted stats by Year/State/Industry
    results = sub.groupby(['year', 'state', 'industry']).apply(
        lambda x: pd.Series({
            'W_Median': (x['median_wage'] * x['total_est_empl']).sum() / x['total_est_empl'].sum(),
            'Jobs': x['total_est_empl'].sum()
        })
    ).reset_index()

    # Convert Jobs to millions for display
    results['Jobs_in_Millions'] = results['Jobs'] / 1_000_000

    # Plotting the evolution
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    sns.lineplot(data=results, x='year', y='W_Median', hue='state', style='industry', markers=True, ax=ax1)
    ax1.set_title("Weighted Median Wage Evolution", fontsize=14)
    ax1.set_ylabel("USD ($)")

    sns.barplot(data=results[results['industry']=='Finance'], x='year', y='Jobs_in_Millions', hue='state', ax=ax2)
    ax2.set_title("Number of Finance Jobs (Millions)", fontsize=14)
    ax2.set_ylabel("Jobs (Millions)")

    plt.tight_layout()
    plt.show()

    # Show numerical table, formatting W_Median as dollars and Jobs as millions
    display(results.sort_values(['year', 'state']).style.format({'W_Median': '${:,.0f}', 'Jobs': '{:,.0f}'}))

state_list = sorted(df['state'].unique())
interact(compare_states,
         state1=widgets.Dropdown(options=state_list, value='NY'),
         state2=widgets.Dropdown(options=state_list, value='IL'))

Cell 5: Geographic Analysis (Maps)

In [ ]:
# Prepare Growth Data (First Year vs Last Year)
first_yr, last_yr = df['year'].min(), df['year'].max()
growth_df = df[df['industry'] == 'Finance'].pivot(index='state', columns='year', values='median_wage')
# Convert Growth to percentage by multiplying by 100
growth_df['Growth'] = ((growth_df[last_yr] - growth_df[first_yr]) / growth_df[last_yr]) * 100

# Create a temporary column for jobs in millions for plotting fig1
df_plotting = df[df['year']==last_yr].copy()
df_plotting['total_est_empl_millions'] = df_plotting['total_est_empl'] / 1_000_000

# 1. Map: Number of Jobs
fig1 = px.choropleth(df_plotting, locations='state', locationmode="USA-states",
                    color='total_est_empl_millions', scope="usa", title=f"Number of Jobs by State (Millions) ({last_yr})",
                    color_continuous_scale="Viridis", labels={'total_est_empl_millions':'Total Jobs (Millions)'})
fig1.show()

# 2. Map: Median Wage
fig2 = px.choropleth(df[df['year']==last_yr], locations='state', locationmode="USA-states",
                    color='median_wage', scope="usa", title=f"Finance Median Wage by State ({last_yr})",
                    color_continuous_scale="RdBu", labels={'median_wage':'Wage ($)'})
fig2.show()

# 3. Map: Wage Growth
fig3 = px.choropleth(growth_df.reset_index(), locations='state', locationmode="USA-states",
                    color='Growth', scope="usa", title=f"Finance Wage Growth ({first_yr} to {last_yr})",
                    color_continuous_scale="RdYlGn", labels={'Growth':'% Change'})
fig3.show()

Cell 6: Finance Premium and Top 10 Tables

In [ ]:
agg_stats = df.groupby('industry').apply(
    lambda x: (x['median_wage'] * x['total_est_empl']).sum() / x['total_est_empl'].sum(),
    include_groups=False
)
premium_usd = agg_stats['Finance'] - agg_stats['Other']
premium_pct = premium_usd / agg_stats['Other']

print(f"--- GLOBAL AGGREGATE PREMIUM ---")
print(f"Dollar Premium: ${premium_usd:,.2f}")
print(f"Percentage Premium: {premium_pct:.2%}\n")

# B. Top 10 Analysis (Latest Year)
latest = df[df['year'] == last_yr].copy()
fin_latest = latest[latest['industry'] == 'Finance'].copy()

# Merging with 'Other' to get state-specific premiums
other_latest = latest[latest['industry'] == 'Other'][['state', 'median_wage']]
fin_latest = fin_latest.merge(other_latest, on='state', suffixes=('_fin', '_oth'))
fin_latest['Premium_$'] = fin_latest['median_wage_fin'] - fin_latest['median_wage_oth']

# Generate Cohorts
top_wage = fin_latest.nlargest(10, 'median_wage_fin')[['state', 'median_wage_fin']]
# Convert top_jobs total_est_empl to millions and format
top_jobs = fin_latest.nlargest(10, 'total_est_empl')[['state', 'total_est_empl']].copy()
top_jobs['total_est_empl_millions'] = top_jobs['total_est_empl'] / 1_000_000

# Growth calculations for Top/Bottom 5%
growth_sorted = growth_df['Growth'].sort_values()
top_growth = growth_sorted.tail(5)
bot_growth = growth_sorted.head(5)

print("TOP 10 FINANCE WAGES:")
display(top_wage.style.format({'median_wage_fin': '${:,.0f}'}))
print("\nTOP 10 FINANCE JOBS (Millions):")
display(top_jobs[['state', 'total_est_empl_millions']].style.format({'total_est_empl_millions': '{:,.0f}M'}))
print("\nTOP 5% WAGE GROWTH STATES:")
display(top_growth.apply(lambda x: f'{x:,.2f}%'))
print("\nBOTTOM 5% WAGE DECLINE/STAGNATION:")
display(bot_growth.apply(lambda x: f'{x:,.2f}%'))